# Évaluation CL — TinyOL — Dataset 3 PRONOSTIA — by_condition

| Champ | Valeur |
|-------|--------|
| **Modèle** | TinyOL (autoencoder 13→16→16→16 gelé + tête OtO 10 params) |
| **Dataset** | FEMTO PRONOSTIA — roulements à billes industriels |
| **Scénario** | by_condition : Condition 1 → Condition 2 → Condition 3 (3 tâches) |
| **Expérience** | exp_052 — voir experiments/exp_052_tinyol_pronostia_by_condition/config_snapshot.yaml |
| **Sprint** | 10 — S10-06 |

> **Modèle supervisé** : TinyOL combine un autoencoder gelé + une tête OtO (One-to-One) entraînable en ligne.  
> Sortie = probabilité de défaut ŷ ∈ [0, 1] (sigmoid).  
> RAM = 3 698 B — très faible empreinte mémoire.  
> Stratégie CL : architecture-based — seule la tête (10 params) est mise à jour sample-by-sample.  
> **Gap 1** : résultat CL TinyOL sur données industrielles réelles de roulements (PRONOSTIA).

```bash
jupyter nbconvert --to notebook --execute \
    notebooks/cl_eval/pronostia_by_condition/tinyol.ipynb \
    --output /tmp/tinyol_pronostia_executed.ipynb --ExecutePreprocessor.timeout=600
```

In [ ]:
# Section 1 — Setup & imports
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from IPython.display import Image, Markdown, display

# --- CWD navigation : notebook 3 niveaux de profondeur ---
_cwd = Path(".").resolve()
if _cwd.name == "pronostia_by_condition":
    os.chdir(_cwd.parent.parent.parent)
elif _cwd.name == "cl_eval":
    os.chdir(_cwd.parent.parent)
elif _cwd.name == "notebooks":
    os.chdir(_cwd.parent)
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.plots import (
    plot_accuracy_matrix,
    plot_confusion_matrix_grid,
    plot_forgetting_curve,
    plot_roc_curves_per_task,
    save_figure,
)
from src.evaluation.feature_space_plots import fit_pca2d, plot_feature_space_2d

# --- Chemins ---
EXP_DIR     = REPO_ROOT / "experiments/exp_052_tinyol_pronostia_by_condition/results"
FIGURES_DIR = REPO_ROOT / "notebooks/figures/cl_evaluation/tinyol/pronostia/by_condition"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

BACKBONE_CKPT   = REPO_ROOT / "experiments/exp_052_tinyol_pronostia_by_condition/backbone.pt"
CONFIG_PATH     = REPO_ROOT / "configs/tinyol_pronostia_by_condition_config.yaml"
NPY_DIR         = REPO_ROOT / "data/raw/Pronostia dataset/binaries"
NORMALIZER_PATH = REPO_ROOT / "configs/pronostia_normalizer.yaml"

# --- Constantes ---
TASK_NAMES     = ["Condition 1 (1800rpm, 4000N)", "Condition 2 (1650rpm, 4200N)", "Condition 3 (1500rpm, 5000N)"]
MODEL_NAME     = "TinyOL"
DATA_AVAILABLE = NPY_DIR.exists() and NORMALIZER_PATH.exists() and BACKBONE_CKPT.exists()

print(f"REPO_ROOT         : {REPO_ROOT}")
print(f"EXP_DIR           : {EXP_DIR}")
print(f"FIGURES_DIR       : {FIGURES_DIR}")
print(f"NPY disponible    : {NPY_DIR.exists()}")
print(f"Backbone .pt      : {BACKBONE_CKPT.exists()}")
print(f"Config disponible : {CONFIG_PATH.exists()}")
print(f"Date exécution    : {datetime.now():%Y-%m-%d %H:%M}")

if not DATA_AVAILABLE:
    display(Markdown(
        "> ⚠️ **Ressources absentes** — Sections 5, 6, 7, 8 en mode dégradé (données synthétiques). "
        "Placer les fichiers NPY, backbone.pt et config pour le mode complet."
    ))

In [ ]:
# Section 2 — Chargement des résultats exp_052
# Spécificité TinyOL : metrics.json a une structure plate et acc_matrix est ragged

metrics_path = EXP_DIR / "metrics.json"
metrics      = json.loads(metrics_path.read_text())

# Reconstruire la matrice 3×3 avec NaN dans le triangle supérieur
# acc_matrix ragged : rows de longueurs 1, 2, 3
acc_matrix_raw = metrics["acc_matrix"]
n_tasks = len(acc_matrix_raw)
acc_matrix_json = np.full((n_tasks, n_tasks), np.nan, dtype=float)
for i, row in enumerate(acc_matrix_raw):
    for j, v in enumerate(row):
        acc_matrix_json[i, j] = v

aa  = metrics["acc_final"]
af  = metrics["avg_forgetting"]
bwt = metrics["backward_transfer"]
ram_b = metrics["ram_peak_bytes"]
lat   = metrics["inference_latency_ms"]
n_encoder = metrics.get("n_params_encoder", 532)
n_oto     = metrics.get("n_params_oto", 10)
n_params  = n_encoder + n_oto

print("=" * 55)
print(f"  Modèle         : TinyOL (autoencoder gelé + tête OtO)")
print(f"  AA             = {aa:.4f}")
print(f"  AF             = {af:.4f}")
print(f"  BWT            = {bwt:+.4f}")
print(f"  RAM peak       = {ram_b} B ({ram_b/1024:.2f} Ko)")
print(f"  Latence        = {lat:.5f} ms")
print(f"  n_params       = {n_params}  (encodeur={n_encoder} + OtO={n_oto})")
print(f"  Budget 64 Ko   : {ram_b <= 65536}")
print("=" * 55)
print("\nMatrice acc (3×3) :")
print(acc_matrix_json)

In [ ]:
# Section 3 — Matrice d'accuracy (heatmap)

fig = plot_accuracy_matrix(
    acc_matrix_json,
    task_names=TASK_NAMES,
    title=f"{MODEL_NAME} — pronostia/by_condition",
)
save_figure(fig, FIGURES_DIR / "acc_matrix.png")
display(Image(str(FIGURES_DIR / "acc_matrix.png")))

In [ ]:
# Section 4 — Courbe d'oubli par tâche

fig = plot_forgetting_curve(
    acc_matrix_json,
    task_names=TASK_NAMES,
    title=f"{MODEL_NAME} — Évolution accuracy par tâche (pronostia/by_condition)",
)
save_figure(fig, FIGURES_DIR / "forgetting_curve.png")
display(Image(str(FIGURES_DIR / "forgetting_curve.png")))

In [ ]:
# Section 5 — Rejeu du scénario CL (collecte preds_dict + proba_dict)
# Reproduit exp_052 : charge le backbone pré-entraîné (backbone.pt),
# instancie une tête OtO fraîche, apprentissage online (batch_size=1, SGD lr=0.01)

preds_dict  = {}  # (i, j) → (y_true, y_pred_binary)
proba_dict  = {}  # (i, j) → sigmoid outputs float32 (pour ROC)
X_tests_raw = []  # [N_val, 13] par tâche — pour la viz PCA
y_tests_raw = []  # [N_val] par tâche

if DATA_AVAILABLE:
    from src.data.pronostia_dataset import get_pronostia_dataloaders
    from src.models.tinyol.autoencoder import TinyOLAutoencoder
    from src.models.tinyol.oto_head import OtOHead, TinyOLOnlineTrainer
    from src.utils.reproducibility import set_seed

    set_seed(42)

    # Charger la config
    if CONFIG_PATH.exists():
        config = yaml.safe_load(CONFIG_PATH.read_text())
    else:
        # Fallback config depuis config_snapshot.yaml
        snap = yaml.safe_load(
            (REPO_ROOT / "experiments/exp_052_tinyol_pronostia_by_condition/config_snapshot.yaml").read_text()
        )
        config = snap

    encoder_dims = tuple(config.get("backbone", {}).get("encoder_dims", [16, 16, 16]))
    decoder_dims = tuple(config.get("backbone", {}).get("decoder_dims", [16, 16, 13]))
    input_dim    = config.get("backbone", {}).get("input_dim", 13)

    autoencoder = TinyOLAutoencoder(
        input_dim=input_dim,
        encoder_dims=encoder_dims,
        decoder_dims=decoder_dims,
    )
    autoencoder.load_state_dict(torch.load(BACKBONE_CKPT, map_location="cpu"))
    autoencoder.freeze_encoder()

    oto_input_dim = config.get("oto_head", {}).get("input_dim", encoder_dims[-1] + 1)
    oto_head = OtOHead(input_dim=oto_input_dim)
    trainer = TinyOLOnlineTrainer(autoencoder, oto_head, config)

    tasks = get_pronostia_dataloaders(
        npy_dir=NPY_DIR,
        normalizer_path=NORMALIZER_PATH,
        batch_size=1,
        seed=42,
    )

    for t in tasks:
        X_v = np.concatenate([b[0].numpy() for b in t["val_loader"]])
        y_v = np.concatenate([b[1].numpy().flatten() for b in t["val_loader"]])
        X_tests_raw.append(X_v)
        y_tests_raw.append(y_v)

    def predict_all(X_np):
        probas = np.zeros(len(X_np), dtype=np.float32)
        for idx, sample in enumerate(X_np):
            p, _ = trainer.predict(torch.from_numpy(sample).float())
            probas[idx] = p
        preds = (probas >= 0.5).astype(float)
        return probas, preds

    for i, task in enumerate(tasks):
        domain = task.get("domain", f"condition_{task['task_id']}")
        print(f"\n--- Tâche {i + 1}/3 : {domain} ---")

        losses = []
        for x_batch, y_batch in task["train_loader"]:
            loss = trainer.update(x_batch.squeeze(0), y_batch.squeeze().float())
            losses.append(loss)
        print(f"  Loss moyenne tâche : {np.mean(losses):.4f} (N={len(losses)})")

        for j in range(i + 1):
            probas, y_pred = predict_all(X_tests_raw[j])
            preds_dict[(i, j)] = (y_tests_raw[j], y_pred)
            proba_dict[(i, j)] = probas
            acc = (y_tests_raw[j] == y_pred).mean()
            print(f"  preds_dict[({i},{j})] → N={len(y_tests_raw[j])}, acc={acc:.4f}")

    print(f"\nScénario CL rejoué — {len(preds_dict)} évaluations collectées")

else:
    display(Markdown("> ⚠️ **Mode dégradé** — ressources absentes. preds_dict synthétique depuis acc_matrix."))

    N_SYNTH = 500
    rng = np.random.default_rng(42)
    y_synth = np.concatenate([np.zeros(int(N_SYNTH * 0.9)), np.ones(int(N_SYNTH * 0.1))])

    for i in range(3):
        for j in range(i + 1):
            noise = rng.normal(0, 0.1, len(y_synth))
            probas_synth = np.where(y_synth == 1, 0.70 + noise, 0.30 + noise).clip(0, 1)
            y_pred_synth = (probas_synth >= 0.5).astype(float)
            preds_dict[(i, j)] = (y_synth.copy(), y_pred_synth)
            proba_dict[(i, j)] = probas_synth.astype(np.float32)

    print("preds_dict synthétique créé (mode dégradé)")

In [ ]:
# Section 6 — Matrices de confusion par tâche

fig = plot_confusion_matrix_grid(
    preds_dict,
    task_names=TASK_NAMES,
    model_name=MODEL_NAME,
    threshold=0.5,
)
save_figure(fig, FIGURES_DIR / "confusion_matrix_grid.png")
display(Image(str(FIGURES_DIR / "confusion_matrix_grid.png")))

In [ ]:
# Section 7 — Courbes ROC par tâche

fig = plot_roc_curves_per_task(
    preds_dict,
    scores_dict=proba_dict,
    task_names=TASK_NAMES,
    model_name=MODEL_NAME,
)

fig.text(
    0.5, 0.01,
    f"AA exp_052 = {aa:.4f}  |  AF = {af:.4f}  |  BWT = {bwt:+.4f}",
    ha="center", fontsize=9, color="#444444",
)

save_figure(fig, FIGURES_DIR / "roc_curves.png")
display(Image(str(FIGURES_DIR / "roc_curves.png")))

In [ ]:
# Section 8 — Espace des features (PCA 2D)

if DATA_AVAILABLE and len(X_tests_raw) == 3:
    X_all      = np.concatenate(X_tests_raw, axis=0)
    y_all      = np.concatenate(y_tests_raw, axis=0)
    domain_ids = np.concatenate([
        np.full(len(X_tests_raw[k]), k) for k in range(3)
    ])

    pca, X_proj = fit_pca2d(X_all)
    expl_var = pca.explained_variance_ratio_
    xlabel = f"PC1 ({expl_var[0]*100:.1f}%)"
    ylabel = f"PC2 ({expl_var[1]*100:.1f}%)"

    fig, ax = plt.subplots(figsize=(9, 7))

    plot_feature_space_2d(
        X_proj, y_all,
        title=f"{MODEL_NAME} — Espace features PCA 2D — pronostia/by_condition",
        ax=ax,
        domain_ids=domain_ids,
        alpha=0.25,
        s=6,
        xlabel=xlabel,
        ylabel=ylabel,
    )

    TASK_COLORS = ["#2196F3", "#FF9800", "#9C27B0"]
    for k, (name, color) in enumerate(zip(TASK_NAMES, TASK_COLORS)):
        mask = domain_ids == k
        cx, cy = X_proj[mask, 0].mean(), X_proj[mask, 1].mean()
        ax.annotate(
            f"C{k+1}",
            xy=(cx, cy),
            fontsize=11,
            fontweight="bold",
            color=color,
            ha="center",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7),
        )

    fig.tight_layout()
    save_figure(fig, FIGURES_DIR / "feature_space_pca.png")
    display(Image(str(FIGURES_DIR / "feature_space_pca.png")))

else:
    display(Markdown(
        "> ⚠️ **Section 8 ignorée** — ressources absentes. "
        "feature_space_pca.png non généré."
    ))
    print("[SKIP] feature_space_pca.png — données non disponibles.")

## Discussion — Gap 1

Ces résultats sur FEMTO PRONOSTIA (données industrielles réelles de roulements à billes)
complètent les validations sur les datasets Kaggle (Equipment Monitoring, Pump Maintenance).

**Comparaison cross-dataset** :
- Dataset Monitoring (by_equipment, 3 tâches) : AA ≈ 0.9123, AF ≈ 0.0079
- **Dataset PRONOSTIA (by_condition, 3 tâches) : AA = 0.9297, AF = 0.0198** ← exp_052

**Particularités TinyOL sur PRONOSTIA** :
- 13 features (vs. 4 sur monitoring) → backbone plus large (encoder 532 params)
- Tête OtO : 10 params seulement — mise à jour sample-by-sample sans momentum
- RAM = 3 698 B — dans le budget 64 Ko STM32N6

`FIXME(gap1)` : ✅ Résultat CL TinyOL sur données industrielles réelles de roulements —
voir `docs/roadmap_phase1.md` section Sprint 10 pour la synthèse complète.

`TODO(arnaud)` : La section Discussion Gap 1 doit-elle inclure une référence bibliographique
au protocole IEEE PHM 2012 Challenge (`@PHM2012`) pour contextualiser dans la littérature ?

In [ ]:
# Section 10 — Tableau récapitulatif + critères d'acceptation

ram_ko = ram_b / 1024

display(Markdown("### Résultats finaux — TinyOL — pronostia/by_condition (exp_052)"))

recap_table = f"""
| Modèle | AA ↑ | AF ↓ | BWT | RAM ↓ | Latence ↓ | n_params |
|--------|------|------|-----|-------|-----------|----------|
| {MODEL_NAME} | {aa:.4f} | {af:.4f} | {bwt:+.4f} | {ram_ko:.2f} Ko | {lat:.5f} ms | {n_params} |
"""
display(Markdown(recap_table))

print("=" * 60)
print("  NOTE SCIENTIFIQUE — Gap 2 (contrainte embarquée STM32N6)")
print("=" * 60)
print(f"  RAM = {ram_b} B = {ram_ko:.2f} Ko")
print(f"  Budget STM32N6 : 65 536 B (64 Ko)")
print(f"  Marge disponible : {65536 - ram_b} B ({(65536 - ram_b)/1024:.1f} Ko)")
print(f"  TinyOL occupe {ram_b / 65536 * 100:.2f}% du budget RAM")
print(f"  Backbone gelé ({n_encoder} params) → stable en Flash")
print(f"  Tête OtO ({n_oto} params, {n_oto*4} B) → seule partie entraînable online")
print()

print("=" * 60)
print("  Critères d'acceptation (S10-06)")
print("=" * 60)
for fig_name in ["acc_matrix.png", "forgetting_curve.png", "confusion_matrix_grid.png",
                 "roc_curves.png", "feature_space_pca.png"]:
    status = "OK" if (FIGURES_DIR / fig_name).exists() else "MANQUANTE"
    print(f"  [{status}] {fig_name}")

print()
print(f"  [{'OK' if abs(aa - 0.9297) < 0.005 else 'WARN'}] AA     = {aa:.4f}  (attendu ≈ 0.9297)")
print(f"  [{'OK' if abs(af - 0.0198) < 0.005 else 'WARN'}] AF     = {af:.4f}  (attendu ≈ 0.0198)")
print(f"  [{'OK' if abs(bwt + 0.0198) < 0.005 else 'WARN'}] BWT    = {bwt:+.4f} (attendu ≈ -0.0198)")
print(f"  [{'OK' if ram_b <= 65536 else 'FAIL'}] RAM    = {ram_ko:.2f} Ko (contrainte ≤ 64 Ko)")
print(f"  [{'OK' if lat < 100.0 else 'WARN'}] Latence= {lat:.5f} ms (contrainte ≤ 100 ms)")